# 05 — Диагностика завершённого эксперимента

Этот notebook ничего не обучает и не изменяет официальный результат. Он читает
диагностические таблицы, сохранённые `04_experiment.ipynb` для выбранного
reference и одного primary candidate.


## 1. Загрузить диагностику выбранного эксперимента

Если каталога нет, выберите эксперимент в `experiment_config.py` и выполните
`04_experiment.ipynb` сверху вниз.


In [ ]:
import json
from pathlib import Path
import sys

from IPython.display import Image, display
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").exists() and (candidate / "src").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.experiment_config import EXPERIMENT_MODULE
import ml_project.experiment as experiment_tools

definition = experiment_tools.load_experiment(EXPERIMENT_MODULE)
settings = definition.settings
diagnostics_dir = (
    PROJECT_ROOT
    / settings.artifact_dir
    / settings.run_name
    / "diagnostics"
)
figure_dir = (
    PROJECT_ROOT
    / "assets/experiments"
    / settings.experiment_id
    / "diagnostics"
)
summary_path = diagnostics_dir / "summary.json"
if not summary_path.exists():
    raise FileNotFoundError(
        f"Нет диагностики для {settings.experiment_id}: {summary_path}. "
        "Сначала выполните 04_experiment.ipynb."
    )

summary = json.loads(summary_path.read_text(encoding="utf-8"))


def read_diagnostic_table(path):
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


tables = {
    path.stem: read_diagnostic_table(path)
    for path in sorted(diagnostics_dir.glob("*.csv"))
}
print("Эксперимент:", settings.experiment_id, settings.experiment_title)
print("Reference:", summary["reference_model"])
print("Candidate:", summary["candidate_model"])
print("Добавлено:", summary["focus_features"] or "нет")
print("Исключено:", summary["removed_features"] or "нет")
print("Каталог:", diagnostics_dir.relative_to(PROJECT_ROOT))


## 2. Проследить признак по pipeline

Здесь проверяем, дошёл ли новый признак до transformed-матрицы, во что он
превратился и не стал ли константным. Preview содержит только focus-колонки;
если feature plan не изменился — первые двадцать transformed-колонок.


In [ ]:
display(tables["pipeline_stages"])
transformed = tables["transformed_features"]
focus_transformed = transformed[
    transformed["focus_feature"].astype(str).str.lower().eq("true")
]
display(focus_transformed if not focus_transformed.empty else transformed.head(30))
display(tables["transformed_preview"].head(30))


## 3. Проверить влияние на folds и importance

`improvement > 0` всегда означает улучшение candidate. Permutation importance
считается только на validation-частях. Native importance — коэффициенты или
встроенная importance estimator, если модель её предоставляет.


In [ ]:
display(tables["paired_fold_deltas"].round(4))

permutation = tables["permutation_importance"]
permutation_summary = (
    permutation.groupby("feature", as_index=False)
    .agg(
        mean_importance=("importance_mean", "mean"),
        std_importance=("importance_mean", "std"),
    )
    .sort_values("mean_importance", ascending=False)
)
display(permutation_summary.round(4))

native = tables["native_importance"]
if native.empty:
    print("Estimator не предоставляет native importance.")
else:
    native_summary = (
        native.groupby(
            ["transformed_feature", "source_feature", "importance_kind"],
            as_index=False,
        )
        .agg(
            mean_value=("value", "mean"),
            std_value=("value", "std"),
            mean_absolute=("absolute_value", "mean"),
        )
        .sort_values("mean_absolute", ascending=False)
    )
    display(native_summary.head(40).round(4))


## 4. Разобрать изменившиеся OOF-предсказания

`fixed` — reference ошибся, candidate исправил ошибку. `broken` — reference
был прав, а candidate создал новую ошибку. Эти строки особенно полезны для
формулировки следующей гипотезы.


In [ ]:
display(tables["prediction_changes"].round(4))
display(tables["confusion"])

OUTCOME_FILTER = ["fixed", "broken"]
oof = tables["oof_predictions"]
changed = oof[oof["outcome_change"].isin(OUTCOME_FILTER)].copy()
if "probability_delta" in changed:
    changed["absolute_probability_delta"] = changed["probability_delta"].abs()
    changed = changed.sort_values("absolute_probability_delta", ascending=False)
display(changed.head(50))


## 5. Проверить threshold, slices и сохранённые графики

Slice-таблица строится только для focus-признаков и групп минимум из десяти
строк. Это исследовательская подсказка, а не новая подтверждённая гипотеза.


In [ ]:
threshold = tables["threshold_metrics"]
if threshold.empty:
    print("Threshold-анализ недоступен для этой модели/задачи.")
else:
    display(threshold[threshold["threshold"].sub(0.5).abs() < 1e-9].round(4))

slices = tables["slice_metrics"]
if slices.empty:
    print("Подходящих focus-slices нет.")
else:
    display(slices.sort_values(["feature", "accuracy_delta"]).round(4))

for figure_path in sorted(figure_dir.glob("*.png")):
    print(figure_path.name)
    display(Image(filename=str(figure_path)))


## 6. Зафиксировать вывод

Перенесите интерпретацию в ручной раздел карточки эксперимента. Завершённый
эксперимент не переделывайте: диагностический вывод должен стать новой
pre-registered гипотезой.
